# FineWeb Cleaning Workflow

**Purpose**: Apply targeted cleaning to FineWeb parquet slices before fine-tuning.

**Scope (Phase 1 — Pre-Easter)**:
1. **Intra-document line deduplication** — remove repeated lines within each document (nav bars, footers, repeated headers)
2. **Boilerplate stripping** — remove cookie banners, privacy notices, navigation fragments
3. **Cross-file URL deduplication** — ensure no URL appears more than once across all slices

**Why these three?** Our quality assessment found that **57.7% of sampled rows** had high character repetition. Most of this comes from repeated template lines within documents, not from truly duplicate documents. Boilerplate and URL duplicates are secondary but easy wins.

**Design**: Process one row group at a time (~1000 rows) to keep RAM constant at ~100MB regardless of file size.

## 1. Setup

In [ ]:
import os
import re
import time
from pathlib import Path
from collections import Counter

import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Auto-detect data directory ---
_candidates = [Path('/data/fineweb'), Path('data/fineweb'), Path('.')]
DATA_DIR = None
for d in _candidates:
    try:
        if d.exists() and list(d.glob('*.parquet')):
            DATA_DIR = d
            break
    except Exception:
        pass

if DATA_DIR is None:
    raise FileNotFoundError('No parquet files found.')

ALL_FILES = sorted(DATA_DIR.glob('*.parquet'))
print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Available slices ({len(ALL_FILES)}):')
for i, f in enumerate(ALL_FILES):
    size_mb = f.stat().st_size / (1024**2)
    pf = pq.ParquetFile(f)
    est_rows = sum(pf.metadata.row_group(j).num_rows for j in range(pf.num_row_groups))
    print(f'  [{i:>2}] {f.name:30s}  {size_mb:>7,.1f} MB  |  ~{est_rows:>8,} rows')

## 2. Select Slices to Clean

Choose which files to process. Set `SELECTED` to a list of indices, or `'all'` for everything.

In [ ]:
# === CONFIGURE HERE ===

# Option A: clean specific slices by index
# SELECTED = [0, 1, 2]

# Option B: clean all slices
SELECTED = 'all'

# Output directory for cleaned files
OUTPUT_DIR = Path('cleaned')

# === Resolve selection ===
if SELECTED == 'all':
    FILES = list(ALL_FILES)
else:
    FILES = [ALL_FILES[i] for i in SELECTED]

OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Selected {len(FILES)} file(s) for cleaning')
print(f'Output directory: {OUTPUT_DIR.resolve()}')
for f in FILES:
    print(f'  - {f.name}')

## 3. Cleaning Functions

Three targeted cleaning steps, each documented with rationale.

### 3.1 Intra-Document Line Deduplication

**Problem**: Many web pages contain repeated navigation bars, sidebar links, or footer text that appear as duplicate lines within the same document. This inflates token counts and degrades training quality.

**Approach**: For each document, split into lines, count duplicates. If the duplicate ratio exceeds a threshold, either deduplicate or discard. We preserve line order by keeping only the first occurrence of each line.

In [ ]:
def dedup_lines_in_doc(text, discard_threshold=0.5, dedup_threshold=0.2):
    """Remove duplicate lines within a single document.

    Args:
        text: The document text.
        discard_threshold: If duplicate ratio exceeds this, discard entire doc (returns None).
        dedup_threshold: If duplicate ratio exceeds this (but below discard), remove dup lines.

    Returns:
        Cleaned text, or None if document should be discarded.
    """
    if not text or not text.strip():
        return None

    lines = text.split('\n')
    if len(lines) <= 1:
        return text

    # Count how many lines are duplicates
    line_counts = Counter(line.strip() for line in lines if line.strip())
    total_lines = sum(line_counts.values())
    dup_lines = sum(c - 1 for c in line_counts.values() if c > 1)

    if total_lines == 0:
        return None

    dup_ratio = dup_lines / total_lines

    # Too many duplicates — likely a broken page or extreme template
    if dup_ratio > discard_threshold:
        return None

    # Moderate duplicates — remove them, keep first occurrence
    if dup_ratio > dedup_threshold:
        seen = set()
        deduped = []
        for line in lines:
            stripped = line.strip()
            if not stripped:  # preserve blank lines for readability
                deduped.append(line)
                continue
            if stripped not in seen:
                seen.add(stripped)
                deduped.append(line)
        return '\n'.join(deduped)

    # Below threshold — keep as-is
    return text


# --- Demo ---
demo_text = 'Home\nAbout\nContact\nWelcome to our site.\nThis is the main content.\nHome\nAbout\nContact\nCopyright 2024'
print('Before:')
print(demo_text)
print('\nAfter:')
print(dedup_lines_in_doc(demo_text))

### 3.2 Boilerplate Stripping

**Problem**: Web pages include cookie consent banners, privacy policies, "subscribe" prompts, and navigation links that aren't useful training content.

**Approach**: Pattern-match common boilerplate lines and remove them. We only remove **entire lines** that are predominantly boilerplate to avoid breaking real content.

In [ ]:
# Patterns that match common boilerplate LINES (applied to individual lines)
BOILERPLATE_LINE_PATTERNS = re.compile(
    r'^\s*('
    r'(accept|manage|reject)\s*(all)?\s*cookies?'
    r'|we use cookies'
    r'|this (site|website) uses cookies'
    r'|cookie (policy|settings|preferences)'
    r'|privacy policy'
    r'|terms (of|and) (service|use|conditions)'
    r'|all rights reserved'
    r'|subscribe to (our )?newsletter'
    r'|sign up for (our )?newsletter'
    r'|click here to'
    r'|share (on|this|via)\s*(facebook|twitter|linkedin|x|email)?'
    r'|follow us (on)?'
    r'|©\s*\d{4}'
    r'|powered by\s+\w+'
    r'|skip to (main )?content'
    r'|toggle (navigation|menu)'
    r'|search(\.\.\.)?$'
    r'|log\s*in$'
    r'|sign\s*(in|up)$'
    r'|menu$'
    r')\s*$',
    re.IGNORECASE
)


def strip_boilerplate(text):
    """Remove lines matching boilerplate patterns.

    Only removes lines that are ENTIRELY boilerplate.
    Does not touch lines where boilerplate text is embedded in real content.
    """
    if not text:
        return text

    lines = text.split('\n')
    cleaned = []
    removed_count = 0

    for line in lines:
        if BOILERPLATE_LINE_PATTERNS.match(line):
            removed_count += 1
        else:
            cleaned.append(line)

    result = '\n'.join(cleaned).strip()

    # If stripping removed too much, the doc might be mostly boilerplate
    if len(result) < 50:
        return None

    return result


# --- Demo ---
demo_bp = 'Skip to main content\nWelcome to our blog\nThis is a real article about AI.\nCookie Policy\nFollow us on Twitter\n© 2024 All Rights Reserved'
print('Before:')
print(demo_bp)
print('\nAfter:')
print(strip_boilerplate(demo_bp))

### 3.3 Cross-File URL Deduplication

**Problem**: The same web page can appear in multiple crawl segments or row groups, resulting in duplicate entries across files.

**Approach**: Maintain a global set of seen URLs. When a URL is encountered a second time, skip that row. Process files in order so the first occurrence is always kept.

In [ ]:
class URLDeduplicator:
    """Track seen URLs across all files for global deduplication."""

    def __init__(self):
        self.seen = set()
        self.dup_count = 0

    def is_duplicate(self, url):
        if url in self.seen:
            self.dup_count += 1
            return True
        self.seen.add(url)
        return False

    def dedup_df(self, df):
        """Remove rows with duplicate URLs from a DataFrame."""
        if 'url' not in df.columns:
            return df
        mask = ~df['url'].apply(lambda u: self.is_duplicate(u))
        return df[mask]

    def stats(self):
        return {'unique_urls': len(self.seen), 'duplicates_removed': self.dup_count}


print('URLDeduplicator ready. Will be instantiated in the pipeline below.')

## 4. Combined Pipeline

Applies all three cleaning steps in sequence to each row group.

In [ ]:
# --- Pipeline config ---
PIPELINE_CONFIG = {
    # Step 1: Line dedup
    'line_dedup_discard': 0.5,    # discard doc if >50% lines are duplicates
    'line_dedup_threshold': 0.2,  # dedup lines if >20% are duplicates

    # Step 2: Boilerplate stripping (no threshold — pattern-based)

    # Step 3: URL dedup (global, handled by URLDeduplicator)

    # Post-cleaning filters
    'min_tokens': 50,
    'min_text_length': 100,       # chars after cleaning
    'min_lang_score': 0.65,
}


def clean_text(text, config):
    """Apply text-level cleaning: line dedup + boilerplate strip.
    Returns cleaned text or None (= discard).
    """
    # Step 1: Line dedup
    text = dedup_lines_in_doc(
        text,
        discard_threshold=config['line_dedup_discard'],
        dedup_threshold=config['line_dedup_threshold'],
    )
    if text is None:
        return None

    # Step 2: Boilerplate stripping
    text = strip_boilerplate(text)
    if text is None:
        return None

    return text


def clean_row_group(df, url_dedup, config):
    """Clean a single row group. Returns (cleaned_df, stats_dict)."""
    stats = Counter()
    n_original = len(df)

    # --- Text-level cleaning ---
    cleaned_texts = []
    keep_mask = []
    for text in df['text']:
        result = clean_text(str(text) if pd.notna(text) else '', config)
        if result is None:
            keep_mask.append(False)
            cleaned_texts.append('')
            stats['text_discarded'] += 1
        else:
            keep_mask.append(True)
            cleaned_texts.append(result)

    df = df[keep_mask].copy()
    df['text'] = [t for t, k in zip(cleaned_texts, keep_mask) if k]

    # --- Post-cleaning filters ---
    if 'token_count' in df.columns:
        mask = df['token_count'] >= config['min_tokens']
        stats['too_few_tokens'] = int((~mask).sum())
        df = df[mask]

    if 'language_score' in df.columns:
        mask = df['language_score'] >= config['min_lang_score']
        stats['low_lang_score'] = int((~mask).sum())
        df = df[mask]

    mask = df['text'].str.len() >= config['min_text_length']
    stats['too_short_after_clean'] = int((~mask).sum())
    df = df[mask]

    # --- URL dedup (global) ---
    before_url = len(df)
    df = url_dedup.dedup_df(df)
    stats['url_duplicate'] = before_url - len(df)

    stats['kept'] = len(df)
    stats['removed'] = n_original - len(df)
    stats['original'] = n_original

    return df, stats


print('Pipeline ready.')

## 5. Run Cleaning

Process all selected files. Progress is printed every 20 row groups.

In [ ]:
url_dedup = URLDeduplicator()
all_stats = []  # per-file stats for the report

total_start = time.time()

for fi, filepath in enumerate(FILES):
    pf = pq.ParquetFile(filepath)
    file_stats = Counter()
    file_start = time.time()
    clean_tables = []

    print(f'\n{"="*80}')
    print(f'[{fi+1}/{len(FILES)}] {filepath.name}  ({pf.num_row_groups} groups)')

    for gi in range(pf.num_row_groups):
        df = pf.read_row_group(gi).to_pandas()
        cleaned, stats = clean_row_group(df, url_dedup, PIPELINE_CONFIG)
        file_stats += stats

        if len(cleaned) > 0:
            clean_tables.append(pa.Table.from_pandas(cleaned, preserve_index=False))

        del df, cleaned

        if (gi + 1) % 20 == 0:
            print(f'  ... {gi+1}/{pf.num_row_groups} groups done')

    # Save cleaned file
    output_path = OUTPUT_DIR / filepath.name
    if clean_tables:
        combined = pa.concat_tables(clean_tables)
        pq.write_table(combined, output_path)
        out_size = output_path.stat().st_size / (1024**2)
    else:
        out_size = 0

    elapsed = time.time() - file_start
    in_size = filepath.stat().st_size / (1024**2)

    print(f'  Original : {file_stats["original"]:>8,} rows  ({in_size:,.1f} MB)')
    print(f'  Kept     : {file_stats["kept"]:>8,} rows  ({out_size:,.1f} MB)')
    print(f'  Removed  : {file_stats["removed"]:>8,} rows')
    print(f'  Breakdown: text_discarded={file_stats["text_discarded"]}, '
          f'too_few_tokens={file_stats["too_few_tokens"]}, '
          f'low_lang_score={file_stats["low_lang_score"]}, '
          f'too_short={file_stats["too_short_after_clean"]}, '
          f'url_dup={file_stats["url_duplicate"]}')
    print(f'  Time: {elapsed:.1f}s')

    all_stats.append({
        'file': filepath.name,
        'original_rows': file_stats['original'],
        'kept_rows': file_stats['kept'],
        'removed_rows': file_stats['removed'],
        'text_discarded': file_stats['text_discarded'],
        'too_few_tokens': file_stats['too_few_tokens'],
        'low_lang_score': file_stats['low_lang_score'],
        'too_short_after_clean': file_stats['too_short_after_clean'],
        'url_duplicate': file_stats['url_duplicate'],
        'keep_rate': round(file_stats['kept'] / file_stats['original'] * 100, 1) if file_stats['original'] else 0,
        'input_mb': round(in_size, 1),
        'output_mb': round(out_size, 1),
        'time_sec': round(elapsed, 1),
    })

total_elapsed = time.time() - total_start
print(f'\n{"="*80}')
print(f'All done in {total_elapsed:.1f}s')
print(f'URL dedup global stats: {url_dedup.stats()}')

## 6. Export Cleaning Report (CSV)

In [ ]:
REPORT_DIR = Path('outputs')
REPORT_DIR.mkdir(exist_ok=True)

# --- Per-file cleaning report ---
report_df = pd.DataFrame(all_stats)

# Add totals row
totals = report_df.select_dtypes(include='number').sum()
totals['file'] = 'TOTAL'
totals['keep_rate'] = round(totals['kept_rows'] / totals['original_rows'] * 100, 1)
report_df = pd.concat([report_df, pd.DataFrame([totals])], ignore_index=True)

report_df.to_csv(REPORT_DIR / 'cleaning_report.csv', index=False)
print('cleaning_report.csv')
print(report_df.to_string(index=False))

# --- Before/After comparison ---
print(f'\n--- Summary ---')
t = totals
print(f'Total original rows : {int(t["original_rows"]):>10,}')
print(f'Total kept rows     : {int(t["kept_rows"]):>10,}')
print(f'Total removed       : {int(t["removed_rows"]):>10,}  ({100 - t["keep_rate"]:.1f}%)')
print(f'\nRemoval breakdown:')
print(f'  Text discarded (line dedup + boilerplate) : {int(t["text_discarded"]):>8,}')
print(f'  Too few tokens (<{PIPELINE_CONFIG["min_tokens"]})                  : {int(t["too_few_tokens"]):>8,}')
print(f'  Low language score (<{PIPELINE_CONFIG["min_lang_score"]})              : {int(t["low_lang_score"]):>8,}')
print(f'  Too short after cleaning (<{PIPELINE_CONFIG["min_text_length"]} chars)     : {int(t["too_short_after_clean"]):>8,}')
print(f'  URL duplicates (cross-file)               : {int(t["url_duplicate"]):>8,}')
print(f'\nURL dedup: {url_dedup.stats()}')
print(f'\nCleaned files saved to: {OUTPUT_DIR.resolve()}')

## 7. Verify Cleaned Output

Quick sanity check: sample from the cleaned files and compare with originals.

In [ ]:
# Compare original vs cleaned for the first file
orig_pf = pq.ParquetFile(FILES[0])
clean_path = OUTPUT_DIR / FILES[0].name

if clean_path.exists():
    clean_pf = pq.ParquetFile(clean_path)

    # Read first row group from each
    orig_df = orig_pf.read_row_group(0).to_pandas()
    clean_df = clean_pf.read_row_group(0).to_pandas()

    print(f'File: {FILES[0].name}')
    print(f'Original row group 0: {len(orig_df)} rows')
    print(f'Cleaned  row group 0: {len(clean_df)} rows')

    # Show a before/after example
    if len(clean_df) > 0:
        idx = 0
        print(f'\n--- Cleaned sample (row {idx}) ---')
        print(f'URL: {clean_df.iloc[idx]["url"]}')
        print(f'Tokens: {clean_df.iloc[idx]["token_count"]}')
        print(f'Lang score: {clean_df.iloc[idx]["language_score"]}')
        text = clean_df.iloc[idx]['text']
        print(f'Text ({len(text)} chars):')
        print(text[:800] + ('\n... [truncated]' if len(text) > 800 else ''))
else:
    print(f'No cleaned file found at {clean_path}')

## Next Steps (Post-Easter)

**Phase 2 — Content quality scoring:**
- Character n-gram diversity score
- Sentence structure analysis (avg sentence length, punctuation density)
- Uppercase ratio filtering
- Alphabetic content ratio
- Optional: perplexity scoring with a small language model

**Phase 3 — Near-duplicate detection:**
- MinHash / LSH for fuzzy deduplication across documents
- Jaccard similarity threshold tuning

**Phase 4 — Fine-tuning pipeline integration:**
- Output cleaned text in training-ready format
- Connect to tokenizer and data loader